# Оценка `dist` по кадру и бортовой телеметрии

Проверяем, помогает ли картинка предсказывать лидарный `dist` сверх `relative_alt`. В каждой строке только один кадр на отдельное измерение лидара, поэтому соседние кадры не копируют одну и ту же метку десятки раз.

Сравниваются:

- `relative_alt` без обучения и его робастная линейная калибровка;
- признаки телеметрии;
- признаки изображения из готовой ImageNet ResNet18 (сама свёрточная сеть не дообучается);
- объединённые признаки изображения и телеметрии, включая модель поправки к `relative_alt`.

Проверка — по очереди оставляем целиком один полёт для теста. При двух полётах это только первый перенос между записями, а не окончательная оценка обобщения. Лидарный `dist=0.1 м` отмечен отдельно: пока неизвестно, является ли это нижним пределом датчика.

In [ ]:
from pathlib import Path
import sys, zipfile, shutil, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), 'Для этой ячейки нужно включить GPU в Kaggle.'
device=torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)

In [ ]:
# Найти CSV и архив изображений в подключённом приватном Kaggle Dataset
input_root=Path('/kaggle/input')
label_paths=list(input_root.rglob('labels.csv'))
if not label_paths: raise FileNotFoundError('В Input не найден labels.csv')
source_root=label_paths[0].parent
work_root=Path('/kaggle/working/aerial_lidar_pairs')
work_root.mkdir(parents=True,exist_ok=True)
image_archives=list(source_root.rglob('images.zip'))
if image_archives:
    with zipfile.ZipFile(image_archives[0]) as z: z.extractall(work_root)
image_paths=[p for p in work_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'}]
if not image_paths:
    image_paths=[p for p in source_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'}]
image_by_name={p.name:p for p in image_paths}
data=pd.read_csv(label_paths[0])
data['lidar_datetime']=pd.to_datetime(data['lidar_datetime'])
for col in ['label_dist_m','relative_alt','roll','pitch','yaw_sin','yaw_cos','xacc','yacc','zacc','camera_offset_s','fc_offset_s']:
    data[col]=pd.to_numeric(data[col],errors='coerce')
data['label_at_sensor_floor']=data['label_at_sensor_floor'].astype(str).str.lower().eq('true')
data['large_jump']=data['large_jump'].astype(str).str.lower().eq('true')
data['image_path']=data['image_filename'].map(image_by_name)
data=data.dropna(subset=['image_path','label_dist_m','relative_alt']).sort_values('lidar_datetime').reset_index(drop=True)
print('Строк с изображением:',len(data),'| по полётам:',data.groupby('flight').size().to_dict())
print('dist=0.1 м:',int(data.label_at_sensor_floor.sum()),'| флаг скачка:',int(data.large_jump.sum()))
print(data[['label_dist_m','relative_alt','camera_offset_s','fc_offset_s']].describe().round(3))

In [ ]:
# Получить 512-мерные визуальные признаки готовой ResNet18
weights=ResNet18_Weights.DEFAULT
transform=weights.transforms()
encoder=resnet18(weights=weights)
encoder.fc=nn.Identity()
encoder=encoder.to(device).eval()

class FrameDataset(Dataset):
    def __init__(self, paths, transform): self.paths=paths; self.transform=transform
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im: x=self.transform(im.convert('RGB'))
        return x

loader=DataLoader(FrameDataset(data.image_path.tolist(),transform),batch_size=64,shuffle=False,num_workers=2,pin_memory=True)
features=[]; t0=time.time()
with torch.inference_mode():
    for batch in loader:
        features.append(encoder(batch.to(device,non_blocking=True)).float().cpu().numpy())
image_features=np.concatenate(features).astype(np.float32)
print('Image features:',image_features.shape,'elapsed_sec:',round(time.time()-t0,1))

In [ ]:
# Leave-one-flight-out: никакие строки тестового полёта не участвуют в обучении
telemetry_cols=['relative_alt','roll','pitch','yaw_sin','yaw_cos','xacc','yacc','zacc']
tab=data[telemetry_cols].replace([np.inf,-np.inf],np.nan)
# Заполнять пропуски медианой обучающего полёта — без использования тестовых данных.

def fit_ridge_with_time_validation(X, target, times):
    order=np.argsort(pd.to_datetime(times).astype('int64').to_numpy())
    cut=max(2,int(len(order)*0.8))
    tr,va=order[:cut],order[cut:]
    if len(va)==0: va=tr[-1:]; tr=tr[:-1]
    best_alpha=None; best_mae=np.inf
    for alpha in [0.1,1.0,10.0,100.0,1000.0]:
        model=make_pipeline(StandardScaler(),Ridge(alpha=alpha))
        model.fit(X[tr],target[tr])
        err=mean_absolute_error(target[va],model.predict(X[va]))
        if err<best_mae: best_alpha,best_mae=alpha,err
    model=make_pipeline(StandardScaler(),Ridge(alpha=best_alpha))
    model.fit(X,target)
    return model,best_alpha

def clip_depth(pred): return np.maximum(np.asarray(pred,dtype=float),0.1)

predictions=[]
for held_out in sorted(data.flight.unique()):
    test_mask=data.flight.eq(held_out).to_numpy(); train_mask=~test_mask
    train=data.loc[train_mask].copy(); test=data.loc[test_mask].copy()
    y_train=train.label_dist_m.to_numpy(); y_test=test.label_dist_m.to_numpy()
    train_alt=train.relative_alt.to_numpy(); test_alt=test.relative_alt.to_numpy()
    x_train=tab.loc[train_mask].copy(); x_test=tab.loc[test_mask].copy()
    med=x_train.median(); x_train=x_train.fillna(med).to_numpy(); x_test=x_test.fillna(med).to_numpy()
    z_train=image_features[train_mask]; z_test=image_features[test_mask]
    x_fused_train=np.column_stack([x_train,z_train]); x_fused_test=np.column_stack([x_test,z_test])
    base_rows=[]
    base_rows.append(('relative_alt_as_is',clip_depth(test_alt),None))
    alt_model=make_pipeline(StandardScaler(),HuberRegressor(epsilon=1.35,alpha=1e-4,max_iter=1000))
    alt_model.fit(train[['relative_alt']],y_train)
    base_rows.append(('huber_relative_alt',clip_depth(alt_model.predict(test[['relative_alt']])),None))
    tel_model=make_pipeline(StandardScaler(),HuberRegressor(epsilon=1.35,alpha=1e-4,max_iter=1000))
    tel_model.fit(x_train,y_train)
    base_rows.append(('huber_telemetry',clip_depth(tel_model.predict(x_test)),None))
    image_model,image_alpha=fit_ridge_with_time_validation(z_train,y_train,train.lidar_datetime)
    base_rows.append(('ridge_image_only',clip_depth(image_model.predict(z_test)),image_alpha))
    fused_model,fused_alpha=fit_ridge_with_time_validation(x_fused_train,y_train,train.lidar_datetime)
    base_rows.append(('ridge_fused_direct',clip_depth(fused_model.predict(x_fused_test)),fused_alpha))
    # Residual fusion asks the image/other sensors to correct the altitude baseline.
    resid_model,resid_alpha=fit_ridge_with_time_validation(x_fused_train,y_train-train_alt,train.lidar_datetime)
    base_rows.append(('ridge_altitude_plus_residual',clip_depth(test_alt+resid_model.predict(x_fused_test)),resid_alpha))
    for model_name,pred,alpha in base_rows:
        part=test[['image_filename','flight','lidar_datetime','label_dist_m','relative_alt','label_at_sensor_floor','large_jump']].copy()
        part['model']=model_name; part['prediction_m']=pred; part['ridge_alpha']=alpha
        predictions.append(part)
pred=pd.concat(predictions,ignore_index=True)

In [ ]:
# Итоговые метрики, отдельно для всех точек и без возможного нижнего порога dist=0.1 м
def metric_row(group, subset):
    y=group.label_dist_m.to_numpy(); p=group.prediction_m.to_numpy()
    return {'subset':subset,'model':group.model.iloc[0],'flight_test':group.flight.iloc[0],
            'n':len(group),'MAE_m':mean_absolute_error(y,p),
            'MedAE_m':median_absolute_error(y,p),'R2':r2_score(y,p)}
metric_rows=[]
for (model,flight),g in pred.groupby(['model','flight']):
    metric_rows.append(metric_row(g,'all'))
    g2=g[~g.label_at_sensor_floor]
    if len(g2): metric_rows.append(metric_row(g2,'excluding_dist_0_1'))
metrics=pd.DataFrame(metric_rows)
display(metrics.round(3))
print('Выбор alpha для ridge:',pred[pred.model.str.startswith('ridge')].groupby(['flight','model']).ridge_alpha.first().to_dict())

In [ ]:
# Сравнение трёх вариантов с телеметрией и кадрами в целом и на малой высоте
focus=['relative_alt_as_is','huber_relative_alt','ridge_image_only','ridge_fused_direct','ridge_altitude_plus_residual']
summary=[]
for model,g in pred[pred.model.isin(focus)].groupby('model'):
    for band,mask in [('all',np.ones(len(g),dtype=bool)),('dist_lt_10m',g.label_dist_m.to_numpy()<10),('dist_10_50m',(g.label_dist_m.to_numpy()>=10)&(g.label_dist_m.to_numpy()<50)),('dist_ge_50m',g.label_dist_m.to_numpy()>=50)]:
        q=g.loc[mask]
        if len(q): summary.append({'model':model,'range':band,'n':len(q),'MAE_m':mean_absolute_error(q.label_dist_m,q.prediction_m),'MedAE_m':median_absolute_error(q.label_dist_m,q.prediction_m)})
summary=pd.DataFrame(summary)
display(summary.round(3))

In [ ]:
# Графики и сохранение артефактов
out=Path('/kaggle/working/aerial_multimodal_results'); out.mkdir(exist_ok=True)
pred.to_csv(out/'predictions.csv',index=False)
metrics.to_csv(out/'metrics_by_fold.csv',index=False)
summary.to_csv(out/'metrics_by_range.csv',index=False)
fig,axes=plt.subplots(2,3,figsize=(16,10))
for ax,model in zip(axes.flat,focus[:6]):
    g=pred[pred.model.eq(model)]
    ax.scatter(g.label_dist_m,g.prediction_m,s=10,alpha=.35)
    lo=min(g.label_dist_m.min(),g.prediction_m.min()); hi=max(g.label_dist_m.max(),g.prediction_m.max())
    ax.plot([lo,hi],[lo,hi],'k--',lw=1)
    ax.set_title(model); ax.set_xlabel('LiDAR dist (m)'); ax.set_ylabel('Prediction (m)'); ax.grid(alpha=.2)
plt.tight_layout(); fig.savefig(out/'lofo_scatter.png',dpi=160)
print('Saved:',list(out.iterdir()))

## Интерпретация

Сравнивайте модели с `relative_alt_as_is`: если признаки изображения и дополнительные параметры не улучшают результат на отложенном полёте, дообучать большую сеть на этих двух записях пока не стоит. Хорошая ошибка `relative_alt` сама по себе не доказывает, что телеметрия измеряет нужную поверхность: это лишь baseline, проверенный на этих двух полётах. Нужны независимые полёты, особенно над рельефом/кронами, и разбор случаев, где `relative_alt` расходится с `dist`.

`R2` зависит от диапазона высот и может выглядеть высоким даже при заметной ошибке в метрах; для малых дистанций смотрите отдельную строку `dist_lt_10m`. Флаги скачков и датчикового порога сохранены в predictions для дополнительного анализа, но автоматически не удаляются.